# Relevance Scores from the Firm Summaries

| Item | Description |
|---|---|
| Goal | Four relevance scores per firm built from the T3.7 summaries of its Item 1 sections, as cosine similarities between a summary embedding and a topic-side embedding: the topic vector (definition or centroid) against (1) the firm summary and (2) each yearly summary, then averaged; the policy summary against (3) the firm summary and (4) each yearly summary, then averaged. |
| Input | `<scratch>/T3.7 - Firm Summaries/Processed/filing_summary_embeddings.npy` + keys and `firm_summary_embeddings.npy` + keys (T3.7.6); `filing_summaries.parquet` and `firm_summaries.parquet` for the texts; `output/T3.5.1/topic_definition.csv` + `topic_definition_embedding.npy`; `output/T3.5.3/nearest_statements.csv` + `topic_centroid_embedding.npy`; `output/T3.5.7/policy_summary_text.csv` + `policy_summary_embedding.npy`; `output/T3.4.1/filings.csv`. |
| Output | `output/T3.8.2/filing_similarities.csv`: one row per filing with its cosine to each topic-side vector. `output/T3.8.2/score_summaries.csv`: one row per firm, topic-side vector, and level. |
| Unit | A firm (gvkey) within the sample; at the yearly level the filing summary is the unit the mean is taken over. |
| Sample | Fiscal years 2010, 2015, 2020 (2,072 filings, 841 firms). The summaries exist only for this sample so far. |
| Version | Prompt version `item1-mapreduce-1.3` of the `summary_lines_of_business` family only (decided 2026-10-25). |
| Example | Meta Platforms, worked through step by step. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.8.2"  # the identifier names this notebook's output folder
TEMP_OUTPUT = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output"
OUTPUT_FOLDER = TEMP_OUTPUT / NOTEBOOK_NAME
PROCESSED_FOLDER = SCRATCH_FOLDER / "T3.7 - Firm Summaries" / "Processed"
TOPIC_SIDE = {  # name -> (embedding file, text file)
    "definition": (TEMP_OUTPUT / "T3.5.1/topic_definition_embedding.npy", TEMP_OUTPUT / "T3.5.1/topic_definition.csv"),
    "centroid": (TEMP_OUTPUT / "T3.5.3/topic_centroid_embedding.npy", TEMP_OUTPUT / "T3.5.3/nearest_statements.csv"),
    "policy_summary": (TEMP_OUTPUT / "T3.5.7/policy_summary_embedding.npy", TEMP_OUTPUT / "T3.5.7/policy_summary_text.csv"),
}
FILINGS_FILE = TEMP_OUTPUT / "T3.4.1/filings.csv"

SET = "three_years"
FAMILY = "summary_lines_of_business"
VERSION = "item1-mapreduce-1.3"  # the only version scored from here on
LEVELS = ["firm_summary", "yearly_mean"]
EXAMPLE_FIRM = "META PLATFORMS INC"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Set {SET}, family {FAMILY}, prompt version {VERSION}")
print(f"Outputs: {OUTPUT_FOLDER}")


Set three_years, family summary_lines_of_business, prompt version item1-mapreduce-1.3
Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.2


## Step 3. The topic side: what the summaries are compared with

Three unit vectors from T3.5. The **definition vector** embeds the codebook label and definition of Topic 68. The **centroid vector** is the mean embedding of the 374 Topic 68 hearing statements, scaled to unit length. The **policy summary vector** embeds the one-paragraph summary of what those statements say the legislation does (T3.5.5, map by hearing year then reduce). All three are printed so the reader sees exactly what each summary is measured against.


In [3]:
topic_vectors = {}
for name, (embedding_file, text_file) in TOPIC_SIDE.items():
    vector = np.load(embedding_file).reshape(-1).astype(np.float32)
    assert vector.shape == (1024,) and abs(np.linalg.norm(vector) - 1) < 1e-4, name
    topic_vectors[name] = vector
definition = pd.read_csv(TOPIC_SIDE["definition"][1]).iloc[0]
print(f"DEFINITION VECTOR = embedding of this text ({definition['n_tokens']} tokens):")
print(f'  "{definition["topic_text"]}"')
nearest = pd.read_csv(TOPIC_SIDE["centroid"][1]).head(3)
print("\nCENTROID VECTOR = mean embedding of the 374 Topic 68 statements; the three statements nearest to it:")
for statement in nearest.itertuples(index=False):
    print(f"  [{statement.hearing_year}, {statement.stance}, cosine to centroid {statement.similarity_to_centroid:.3f}] {statement.text[:300]}{'...' if len(statement.text) > 300 else ''}")
policy = pd.read_csv(TOPIC_SIDE["policy_summary"][1]).iloc[0]
print(f"\nPOLICY SUMMARY VECTOR = embedding of this text ({policy['n_words']} words, {policy['n_tokens']} tokens):")
print(f'  "{policy["policy_summary"]}"')
names = list(topic_vectors)
print("\nCosines among the three topic-side vectors:")
display(pd.DataFrame([[float(topic_vectors[a] @ topic_vectors[b]) for b in names] for a in names], index=names, columns=names).round(3))


DEFINITION VECTOR = embedding of this text (36 tokens):
  "consumer data privacy legislation: This topic encompasses federal and state laws, proposed bills, and regulatory frameworks specifically addressing the collection, use, and protection of consumer personal information."

CENTROID VECTOR = mean embedding of the 374 Topic 68 statements; the three statements nearest to it:
  [2008, concern, cosine to centroid 0.879] I think that the issue here of privacy, who is collecting what kind of information, how does that information exist in what identifiable form or how might that be used, who is it sold to, does the consumer know, all of those issues I think are very, very important. And the issues that surround them ...
  [2019, support, cosine to centroid 0.876] Well, I happen to believe that this is one of the areas, maybe not many areas in this next couple of years, that we ought to be able to come together around in a bipartisan way and come up with a national data privacy law that c

,definition,centroid,policy_summary
definition,1.000,0.745,0.761
centroid,0.745,1.000,0.790
policy_summary,0.761,0.790,1.000


## Step 4. The firm side: the yearly summaries and the firm summary

For each firm in the sample, T3.7.2 wrote one summary per filing (fiscal 2010, 2015, 2020 where the firm filed), and T3.7.4 wrote one firm summary from those yearly summaries. T3.7.6 embedded all of them. Only the chosen version is kept here.


In [4]:
filing_keys = pd.read_parquet(PROCESSED_FOLDER / "filing_summary_embedding_keys.parquet")
filing_vectors = np.load(PROCESSED_FOLDER / "filing_summary_embeddings.npy")
firm_keys = pd.read_parquet(PROCESSED_FOLDER / "firm_summary_embedding_keys.parquet")
firm_vectors = np.load(PROCESSED_FOLDER / "firm_summary_embeddings.npy")
assert len(filing_keys) == len(filing_vectors) and len(firm_keys) == len(firm_vectors)
keep = filing_keys["family"].eq(FAMILY) & filing_keys["prompt_version"].eq(VERSION)
filing_keys, filing_vectors = filing_keys.loc[keep].reset_index(drop=True), filing_vectors[keep.to_numpy()]
keep = firm_keys["set"].eq(SET) & firm_keys["family"].eq(FAMILY) & firm_keys["prompt_version"].eq(VERSION)
firm_keys, firm_vectors = firm_keys.loc[keep].reset_index(drop=True), firm_vectors[keep.to_numpy()]
assert np.allclose(np.linalg.norm(filing_vectors, axis=1), 1, atol=1e-4) and np.allclose(np.linalg.norm(firm_vectors, axis=1), 1, atol=1e-4)

filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
in_set = filings.loc[filings["in_" + SET].eq(1), ["accessionNo", "gvkey", "conm", "fyear"]]
filing_table = filing_keys.merge(in_set, on="accessionNo", how="left", validate="one_to_one")
assert filing_table["gvkey"].notna().all() and len(filing_table) == len(in_set)
firm_names = in_set.drop_duplicates("gvkey").set_index("gvkey")["conm"]
assert set(firm_keys["gvkey"]) == set(firm_names.index)
filing_texts = pd.read_parquet(PROCESSED_FOLDER / "filing_summaries.parquet", columns=["accessionNo", "family", "prompt_version", "summary", "n_words"])
filing_texts = filing_texts.loc[filing_texts["family"].eq(FAMILY) & filing_texts["prompt_version"].eq(VERSION)].set_index("accessionNo")
firm_texts = pd.read_parquet(PROCESSED_FOLDER / "firm_summaries.parquet", columns=["gvkey", "set", "family", "prompt_version", "summary", "n_words"])
firm_texts = firm_texts.loc[firm_texts["set"].eq(SET) & firm_texts["family"].eq(FAMILY) & firm_texts["prompt_version"].eq(VERSION)].set_index("gvkey")
print(f"Yearly summaries: {len(filing_table):,} ({filing_table['gvkey'].nunique():,} firms); firm summaries: {len(firm_keys):,}")


Yearly summaries: 2,072 (841 firms); firm summaries: 841


## Step 5. How the four scores are computed

For firm $i$, let $f_i$ be the embedding of its firm summary, $e_{it}$ the embedding of its yearly summary for fiscal year $t$, $T_i$ the firm's fiscal years in the sample (one to three of 2010, 2015, 2020), and $v$ a topic-side vector. All embeddings are unit length, so the cosine is the dot product.

1. **Topic vector vs the firm summary** (one cosine per firm), for $v$ = definition and $v$ = centroid:
$$\text{score}^{\text{firm}}_{i,v} = \cos(f_i, v)$$

2. **Topic vector vs each yearly summary, then the mean** over the firm's years:
$$\text{score}^{\text{yearly}}_{i,v} = \frac{1}{|T_i|} \sum_{t \in T_i} \cos(e_{it}, v)$$

3. **Policy summary vs the firm summary**: the same as (1) with $v$ = the policy summary embedding.

4. **Policy summary vs each yearly summary, then the mean**: the same as (2) with $v$ = the policy summary embedding.

In words: for the firm level, compare the one firm summary with the topic side; for the yearly level, compare each year's summary with the topic side and average the yearly cosines. The table below carries all six combinations (three topic-side vectors × two levels); (1) and (2) are the definition and centroid rows, (3) and (4) the policy-summary rows.


In [5]:
for name, vector in topic_vectors.items():
    filing_table[f"cos_{name}"] = filing_vectors @ vector
    firm_keys[f"cos_{name}"] = firm_vectors @ vector
cos_columns = [f"cos_{name}" for name in topic_vectors]
yearly_mean = filing_table.groupby("gvkey").agg(n_filings=("accessionNo", "size"), **{c: (c, "mean") for c in cos_columns}).reset_index()
firm_level = firm_keys.merge(yearly_mean[["gvkey", "n_filings"]], on="gvkey", how="left", validate="one_to_one")
rows = []
for level, frame in [("firm_summary", firm_level), ("yearly_mean", yearly_mean)]:
    for name in topic_vectors:
        rows.append(frame[["gvkey", "n_filings", f"cos_{name}"]].rename(columns={f"cos_{name}": "score"}).assign(level=level, topic_vector=name))
scores = pd.concat(rows, ignore_index=True)
scores["conm"] = scores["gvkey"].map(firm_names)
scores["method"], scores["sample"], scores["family"], scores["prompt_version"] = "summaries", SET, FAMILY, VERSION
scores = scores[["gvkey", "conm", "method", "sample", "family", "prompt_version", "topic_vector", "level", "n_filings", "score"]]
assert len(scores) == len(firm_names) * len(LEVELS) * len(topic_vectors)
print(f"Score rows: {len(scores):,} = {len(firm_names):,} firms x {len(topic_vectors)} topic-side vectors x {len(LEVELS)} levels")
display(scores.groupby(["topic_vector", "level"])["score"].describe()[["min", "25%", "50%", "75%", "max"]].round(3))


Score rows: 5,046 = 841 firms x 3 topic-side vectors x 2 levels


min    25%    50%    75%    max
topic_vector   level                                          
centroid       firm_summary  0.419  0.484  0.503  0.521  0.593
               yearly_mean   0.397  0.481  0.500  0.521  0.597
definition     firm_summary  0.309  0.410  0.433  0.453  0.553
               yearly_mean   0.322  0.409  0.434  0.458  0.538
policy_summary firm_summary  0.401  0.488  0.512  0.540  0.622
               yearly_mean   0.387  0.481  0.507  0.533  0.626

## Step 6. Worked example: Meta Platforms

The texts on the firm side, then each yearly cosine, its mean, and the firm-summary cosine for every topic-side vector.


In [6]:
example_gvkey = firm_names.index[firm_names.eq(EXAMPLE_FIRM)]
assert len(example_gvkey) == 1, "the example firm name must match one gvkey"
example_gvkey = example_gvkey[0]
example_filings = filing_table.loc[filing_table["gvkey"].eq(example_gvkey)].sort_values("fyear")
print(f"{EXAMPLE_FIRM} (gvkey {example_gvkey}): {len(example_filings)} yearly summaries, fiscal {example_filings['fyear'].tolist()}")
for filing in example_filings.itertuples(index=False):
    text = filing_texts.loc[filing.accessionNo]
    print(f"\n--- yearly summary, fiscal {filing.fyear} ({text['n_words']} words) ---\n{text['summary']}")
firm_text = firm_texts.loc[example_gvkey]
print(f"\n--- firm summary ({firm_text['n_words']} words) ---\n{firm_text['summary']}")
print("\nCosines:")
table = example_filings.set_index("fyear")[cos_columns].rename(columns=lambda c: c.replace("cos_", ""))
table.index = [f"yearly summary, fiscal {year}" for year in table.index]
table.loc["mean of the yearly cosines  (score, level yearly_mean)"] = table.mean()
table.loc["firm summary  (score, level firm_summary)"] = firm_keys.loc[firm_keys["gvkey"].eq(example_gvkey), cos_columns].iloc[0].rename(lambda c: c.replace("cos_", ""))
display(table.round(4))


META PLATFORMS INC (gvkey 170617): 2 yearly summaries, fiscal [2015, 2020]

--- yearly summary, fiscal 2015 (123 words) ---
META PLATFORMS INC. operates a suite of mobile and web applications designed to enable people to connect, share, discover, and communicate. Its primary products include Facebook, a platform for connecting and sharing content; Instagram, for photo and video sharing; Messenger, for instant messaging; WhatsApp, a global messaging application; and Oculus, a virtual reality technology platform. The company generates revenue by selling advertising placements to marketers who utilize these platforms to reach audiences based on factors such as age, location, and interests. These marketers purchase ads that appear across Facebook, Instagram, and third-party applications. The company's customers consist of individuals who use its products to interact and share information, as well as marketers and businesses seeking to engage with customers through its advertising ecosystem

,definition,centroid,policy_summary
"yearly summary, fiscal 2015",0.4473,0.5377,0.5458
"yearly summary, fiscal 2020",0.4657,0.5502,0.5547
"mean of the yearly cosines (score, level yearly_mean)",0.4565,0.5439,0.5503
"firm summary (score, level firm_summary)",0.4556,0.5364,0.5549


## Step 7. Compare the two levels and the three topic-side vectors

Spearman rank correlations across firms, and the ten highest and lowest firms per vector at the firm-summary level. A first look; the figures in T3.9 do this properly.


In [7]:
wide = scores.pivot_table(index="gvkey", columns=["level", "topic_vector"], values="score")
print("Yearly mean vs firm summary, Spearman by topic-side vector:")
display(pd.Series({name: wide[("yearly_mean", name)].corr(wide[("firm_summary", name)], method="spearman") for name in topic_vectors}).round(3).to_frame("spearman").T)
for level in LEVELS:
    print(f"Among the topic-side vectors at level {level}, Spearman:")
    display(wide[level].corr(method="spearman").round(3))
firm_scores = scores.loc[scores["level"].eq("firm_summary")]
for name in topic_vectors:
    part = firm_scores.loc[firm_scores["topic_vector"].eq(name)].sort_values("score", ascending=False)
    print(f"--- {name} vs the firm summary: ten highest (left) and ten lowest (right)")
    display(pd.concat([part.head(10)[["conm", "score"]].reset_index(drop=True), part.tail(10)[["conm", "score"]].reset_index(drop=True)], axis=1, keys=["highest", "lowest"]).round(3))


Yearly mean vs firm summary, Spearman by topic-side vector:


,definition,centroid,policy_summary
spearman,0.865,0.84,0.893


Among the topic-side vectors at level firm_summary, Spearman:


topic_vector,centroid,definition,policy_summary
topic_vector,,,
centroid,1.000,0.734,0.798
definition,0.734,1.000,0.649
policy_summary,0.798,0.649,1.000


Among the topic-side vectors at level yearly_mean, Spearman:


topic_vector,centroid,definition,policy_summary
topic_vector,,,
centroid,1.000,0.771,0.800
definition,0.771,1.000,0.674
policy_summary,0.800,0.674,1.000


... (2 outputs left out)

--- centroid vs the firm summary: ten highest (left) and ten lowest (right)


highest                               lowest       
                          conm  score                          conm  score
0               RS LEGACY CORP  0.593           VICI PROPERTIES INC  0.442
1         FAMILY DOLLAR STORES  0.588        YUM CHINA HOLDINGS INC  0.441
2              GEN DIGITAL INC  0.581   NORWEGIAN CRUISE LINE HLDGS  0.441
3              FIRST DATA CORP  0.581  DIAMOND OFFSHRE DRILLING INC  0.438
4                GOODRICH CORP  0.576                   BAXALTA INC  0.438
5  GREAT ATLANTIC & PAC TEA CO  0.574     INTL FLAVORS & FRAGRANCES  0.434
6         CONEXANT SYSTEMS INC  0.574     MGM RESORTS INTERNATIONAL  0.433
7       PALO ALTO NETWORKS INC  0.573   ALPHA NATURAL RESOURCES INC  0.427
8                     ADT CORP  0.571    HUNTINGTON INGALLS IND INC  0.427
9                     VISA INC  0.571   ALEXION PHARMACEUTICALS INC  0.419

--- policy_summary vs the firm summary: ten highest (left) and ten lowest (right)


highest                               lowest       
                          conm  score                          conm  score
0   VERIZON COMMUNICATIONS INC  0.622          CLEVELAND-CLIFFS INC  0.438
1       PALO ALTO NETWORKS INC  0.622                JOY GLOBAL INC  0.437
2              GEN DIGITAL INC  0.616  DIAMOND OFFSHRE DRILLING INC  0.435
3                  SPRINT CORP  0.611          KANSAS CITY SOUTHERN  0.433
4               RS LEGACY CORP  0.608   ALPHA NATURAL RESOURCES INC  0.430
5                 BROADCOM INC  0.607        DARDEN RESTAURANTS INC  0.428
6  MOTOROLA MOBILITY HLDGS INC  0.606          CONCHO RESOURCES INC  0.428
7      WINDSTREAM HOLDINGS INC  0.603          FREEPORT-MCMORAN INC  0.424
8              FIRST DATA CORP  0.599   ALEXION PHARMACEUTICALS INC  0.420
9                     VISA INC  0.599    HUNTINGTON INGALLS IND INC  0.401

## Step 8. Save the filing cosines and the firm scores


In [8]:
filing_file = OUTPUT_FOLDER / "filing_similarities.csv"
filing_table[["accessionNo", "gvkey", "conm", "fyear", "family", "prompt_version"] + cos_columns].sort_values(["gvkey", "fyear"]).to_csv(filing_file, index=False, float_format="%.6f")
scores_file = OUTPUT_FOLDER / "score_summaries.csv"
scores.sort_values(["topic_vector", "level", "gvkey"]).to_csv(scores_file, index=False, float_format="%.6f")
print(f"Saved: {filing_file} ({len(filing_table):,} rows)")
print(f"Saved: {scores_file} ({len(scores):,} rows; columns {list(scores.columns)})")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.2/filing_similarities.csv (2,072 rows)
Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.8.2/score_summaries.csv (5,046 rows; columns ['gvkey', 'conm', 'method', 'sample', 'family', 'prompt_version', 'topic_vector', 'level', 'n_filings', 'score'])
